# Hello ABC: From XDOF demonstrations to robot evaluation

Watch a human demonstration, inspect what a VLA predicts, and measure a robot completing a task in simulation. Then optionally fine-tune and compare a new checkpoint.

**Task:** bottles into a bin on the ABC bimanual YAM platform.

**Journey:** Demonstration → observation → predicted action chunk → closed-loop episode → measured outcomes → optional fine-tuning.

- **Laptop path:** a small prepared real/sim preview, inline video, camera views, state/action plots and a provenance export. No model download.
- **GPU path:** released ABC-VLA inference and complete simulator episodes on Linux/NVIDIA. Set `RUN_GPU = True` below. This downloads roughly 8.8 GB of weights plus simulator assets.
- **Fine-tuning:** explicitly opt in after the baseline works. The default 20-step exercise demonstrates training/checkpoint mechanics, not a meaningful specialization claim.

This standalone tutorial calls ABC directly. It does not add training or simulator execution to the ROVE application. No physical robot is controlled.

Start Jupyter using the adjacent [README](README.md). Code is pinned to [ABC d0e987b](https://github.com/amazon-far/abc/tree/d0e987b61b376f1a1b8777b19149f190b2ba6939). The prepared preview is published by ABC from its real/sim data; it is **not** a fresh download of the gated raw [XDOF/ABC-130k](https://huggingface.co/datasets/XDOF/ABC-130k) repository.

In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import uuid
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, Video, display
from PIL import Image

# Supports Jupyter opened at the repository root or beside this notebook.
HERE = Path.cwd()
if not (HERE / "tutorial.py").exists():
    HERE = HERE / "notebooks" / "abc_xdof"
assert (HERE / "tutorial.py").is_file(), (
    "Open this notebook from its folder or the ROVE repository root."
)
sys.path.insert(0, str(HERE))

In [ ]:
from tutorial import (
    ABC_REPOSITORY,
    ABC_REVISION,
    PARENT_NAME,
    TASK,
    comparison_notes,
    data_manifest,
    evaluation_args,
    read_episode,
    read_result,
    run_command,
    sha256,
    tracked_stage,
    write_json,
)

RUN_GPU = False
RUN_FINE_TUNING = False
EVAL_SEEDS = [11, 29, 47]  # A small demonstration, not a benchmark-quality sample.
TRAIN_STEPS = 20
TRAIN_BATCH_SIZE = 1
TRAIN_GPUS = 1  # Use >1 with FSDP when the measured memory budget requires it.
WORK = Path(os.environ.get("ABC_NOTEBOOK_WORK", HERE / ".work")).resolve()
ABC = WORK / "abc"
CACHE = WORK / "preview-cache"
SESSION = WORK / "sessions" / uuid.uuid4().hex[:12]
SESSION.mkdir(parents=True, exist_ok=False)
CACHE.mkdir(parents=True, exist_ok=True)
BASELINE, CANDIDATE = [], []
STAGES = {
    "inference": "not_started" if RUN_GPU else "skipped",
    "evaluation": "not_started" if RUN_GPU else "skipped",
    "fine_tuning": "not_started" if RUN_FINE_TUNING else "skipped",
}
assert not RUN_FINE_TUNING or RUN_GPU, "Fine-tuning requires RUN_GPU=True."
assert EVAL_SEEDS and len(set(EVAL_SEEDS)) == len(EVAL_SEEDS)
assert TRAIN_STEPS > 0 and TRAIN_GPUS > 0 and TRAIN_BATCH_SIZE > 0
print("Outputs:", SESSION)

## 1. Check the environment

Data inspection needs Python 3.12, Git and FFmpeg. The GPU sections use a **separate ABC environment**, so ABC's PyTorch and MuJoCo versions do not replace the notebook's packages or ROVE's dependencies. A notebook can connect to a remote Linux GPU machine; merely opening it on a Mac does not supply a CUDA device.

In [ ]:
checks = {
    "Python 3.12": sys.version_info[:2] == (3, 12),
    "Git": bool(shutil.which("git")),
    "FFmpeg": bool(shutil.which("ffmpeg")),
}
for name, ready in checks.items():
    print(f"{'Ready' if ready else 'Missing'}: {name}")
assert all(checks.values()), (
    "Install the missing prerequisites using README.md, then restart the kernel."
)
if RUN_GPU:
    assert platform.system() == "Linux", "Run the GPU path on a Linux NVIDIA host."
    assert shutil.which("uv") and shutil.which("nvidia-smi"), (
        "GPU path needs uv and NVIDIA driver tools."
    )
    run_command(["nvidia-smi", "-L"], cwd=HERE, log=SESSION / "gpu.log")
else:
    print(
        "Data exploration is enabled. Inference, simulation and training will be explicitly skipped."
    )

## 2. Get ABC and a small task sample

The next cell clones the pinned public source and downloads ABC's prepared preview (about 162 MB compressed (measured for this preview)). It reuses the local cache on subsequent executions. The dedicated cache must contain only this tutorial's preview; do not mix unrelated tasks into its training folders.

The preview includes **real demonstrations and simulation demonstrations in separate folders**. Real recordings train or diagnose policies; the simulator generates new observations and outcomes when evaluating a policy. Recorded video is never treated as a simulator that can respond to new actions.

In [ ]:
if not ABC.exists():
    run_command(
        ["git", "clone", "--filter=blob:none", ABC_REPOSITORY, str(ABC)],
        cwd=WORK,
        log=SESSION / "clone.log",
    )
    run_command(
        ["git", "checkout", "--detach", ABC_REVISION], cwd=ABC, log=SESSION / "checkout.log"
    )
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ABC, text=True).strip()
assert revision == ABC_REVISION, (
    "This checkout differs from the reviewed version; use a fresh ABC_NOTEBOOK_WORK."
)
run_command(
    [sys.executable, "prepare.py", "--cache", str(CACHE)],
    cwd=ABC,
    log=SESSION / "prepare-preview.log",
)
SPLITS = ["train_real", "val_real", "train_sim", "val_sim"]
EPISODES = {split: sorted((CACHE / split).glob("episode_*")) for split in SPLITS}
for split, episodes in EPISODES.items():
    print(f"{split}: {len(episodes)} episodes")
assert all(EPISODES.values()), "Expected real/sim train/validation preview folders."
for domain in ("real", "sim"):
    train_ids = {ep.name for ep in EPISODES[f"train_{domain}"]}
    val_ids = {ep.name for ep in EPISODES[f"val_{domain}"]}
    assert train_ids.isdisjoint(val_ids), "Training and validation share episode IDs."
files = data_manifest(CACHE)
write_json(SESSION / "data-manifest.json", files)
print("Prepared file fingerprints:", len(files))

## 3. Watch a demonstration and inspect the robot inputs

Select a validation episode. Its prepared export contains a stacked camera video and a float64 table: **14 measured state values followed by 14 commanded action targets**. Arm joints are radians; the two gripper values are separate aperture coordinates. Do not mix their units in one error score.

The conversion places these streams on a 30 Hz grid. We select video frames by **frame ordinal**, not wall-clock seeking. Native raw MCAP streams require timestamp alignment before conversion. [Upstream episode format](https://github.com/amazon-far/abc/blob/d0e987b61b376f1a1b8777b19149f190b2ba6939/abc_minimal/README.md#converting-local-mcaps--the-episode-format)

In [ ]:
EPISODE = EPISODES["val_real"][0]
metadata, states, demonstrated_actions = read_episode(EPISODE)
FRAME = min(30, len(states) - 1)
REAL_PROMPT = metadata.get("instruction") or metadata.get("task_name", "").replace("_", " ")
assert REAL_PROMPT, "The demonstration needs a recorded task instruction."
print("Task:", REAL_PROMPT)
print("Episode:", EPISODE.name)
print("States / actions:", states.shape, demonstrated_actions.shape)
print("Camera stack order:", metadata["cameras"])

# Re-time only a short display clip; never use this re-encoded clip for model input.
source_video = EPISODE / "combined_camera-images-rgb.mp4"
clip = SESSION / "demonstration.mp4"
run_command(
    [
        "ffmpeg",
        "-v",
        "error",
        "-y",
        "-i",
        str(source_video),
        "-vf",
        "setpts=N/(30*TB)",
        "-frames:v",
        "180",
        "-r",
        "30",
        "-an",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        str(clip),
    ],
    cwd=HERE,
    log=SESSION / "demo-video.log",
)
display(Video(filename=str(clip), embed=True, width=300))

In [ ]:
frame_file = SESSION / "observation.png"
run_command(
    [
        "ffmpeg",
        "-v",
        "error",
        "-y",
        "-i",
        str(source_video),
        "-vf",
        rf"select=eq(n\,{FRAME})",
        "-frames:v",
        "1",
        str(frame_file),
    ],
    cwd=HERE,
    log=SESSION / "frame.log",
)
stacked = np.asarray(Image.open(frame_file).convert("RGB"))
cameras = metadata["cameras"]
assert stacked.shape[0] % len(cameras) == 0
height = stacked.shape[0] // len(cameras)
fig, axes = plt.subplots(1, len(cameras), figsize=(12, 4), squeeze=False)
for i, name in enumerate(cameras):
    axes[0, i].imshow(stacked[i * height : (i + 1) * height])
    axes[0, i].set_title(name)
    axes[0, i].axis("off")
plt.tight_layout()
plt.show()

seconds = np.arange(len(states)) / 30
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(seconds, states[:, 0], label="Measured joint")
axes[0].plot(seconds, demonstrated_actions[:, 0], label="Commanded joint", alpha=0.7)
axes[0].set(xlabel="Seconds", ylabel="Radians", title="Left arm · joint 1")
axes[1].plot(seconds, states[:, 6], label="Measured gripper")
axes[1].plot(seconds, demonstrated_actions[:, 6], label="Commanded gripper", alpha=0.7)
axes[1].set(xlabel="Seconds", ylabel="Aperture coordinate", title="Left gripper")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Load the released ABC-VLA (GPU option)

Re-run from the configuration cell with `RUN_GPU=True` on a Linux/NVIDIA host. This installs ABC's own environment and downloads its **abc130k step-200000 VLA parent**, including checksum-verified metadata and simulator assets. These Gemma-derived weights have their own [use terms](https://github.com/amazon-far/abc#licenses).

This parent has already seen real and simulation tasks. The exercise measures specialization and behavior on new rollouts; it does not establish unseen-task generalization. Upstream does not establish a minimum evaluation VRAM requirement.

In [ ]:
PARENT = CACHE / PARENT_NAME
ABC_PYTHON = ABC / ".venv" / "bin" / "python"


def abc_command(args, log):
    assert RUN_GPU and ABC_PYTHON.is_file(), "Set up the GPU environment first."
    run_command(
        [str(ABC_PYTHON), *map(str, args)],
        cwd=ABC,
        log=log,
        env={"ABC_CACHE": str(CACHE), "MUJOCO_GL": "egl"},
    )


if RUN_GPU:
    run_command(["uv", "sync", "--python", "3.12"], cwd=ABC, log=SESSION / "abc-install.log")
    abc_command(
        [
            "-c",
            "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name())",
        ],
        SESSION / "cuda.log",
    )
    abc_command(
        [
            "prepare.py",
            "--cache",
            CACHE,
            "--vla-pretrained",
            "--vla-pretrained-family",
            "abc130k",
            "--vla-pretrained-step",
            "200000",
        ],
        SESSION / "prepare-model.log",
    )
    parent_metadata = json.loads(PARENT.with_suffix(".json").read_text())
    PARENT_PROMPT = parent_metadata["sim_prompt_map"][TASK]
    print("Parent checkpoint:", PARENT.name)
    print("Simulator prompt:", PARENT_PROMPT)
else:
    print("Skipped: GPU installation and model download.")

## 5. Predict one action chunk

Only the current observation, robot state and task instruction enter the model. Recorded future actions remain a separate reference. ABC's native decoder handles the prepared video's timing and camera mapping; its VLA adapter handles normalization exactly once.

The short script runs in ABC's environment and exits afterwards, releasing GPU memory before evaluation or training. **Action disagreement is a diagnostic, not a task-success verdict.** Different valid motions can complete the same task.

In [ ]:
prediction_source = r"""
import json, sys
from pathlib import Path
import numpy as np
from abc_minimal.episode_io import load_episode
from abc_minimal.dataloader import decode_frame, _episode_prompt
from abc_minimal.dit import task_name_to_prompt
from abc_minimal.policy import VLAInferencePolicy, VLAPolicyConfig

episode, checkpoint, output, frame = sys.argv[1:]
episode, frame = Path(episode), int(frame)
meta, state, reference = load_episode(episode)
prompt = _episode_prompt(episode, meta, meta["task_name"]) or task_name_to_prompt(meta["task_name"])
config = VLAPolicyConfig(prompt=prompt)
policy = VLAInferencePolicy(Path(checkpoint), config, "cuda")
images = decode_frame(episode, frame, len(state), tuple(meta["cameras"]), policy.camera_keys)
observation = {"state": state[frame].astype(np.float32), "images": images, "prompt": prompt}
noise = np.random.default_rng(0).standard_normal((policy.chunk_length, policy.action_dim), dtype=np.float32)
predicted = policy.infer(observation, noise=noise)
np.savez(output, predicted=predicted, reference=reference[frame:frame+len(predicted)])
print(json.dumps({"prompt": prompt, "action_shape": list(predicted.shape), "units": "joint radians and gripper coordinates"}))
"""
if RUN_GPU:
    with tracked_stage(STAGES, "inference"):
        script = SESSION / "predict.py"
        script.write_text(prediction_source)
        # ABC is an installed package in its own environment; the source script stays beside our artifacts.
        abc_command(
            [script, EPISODE, PARENT, SESSION / "prediction.npz", FRAME], SESSION / "predict.log"
        )
        with np.load(SESSION / "prediction.npz") as data:
            predicted, reference = data["predicted"], data["reference"]
        n = min(len(predicted), len(reference))
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.plot(np.arange(n) / 30, reference[:n, 0], label="Demonstrated action")
        ax.plot(np.arange(n) / 30, predicted[:n, 0], label="Predicted action")
        ax.set(
            title="Left joint 1 · diagnostic comparison", xlabel="Seconds ahead", ylabel="Radians"
        )
        ax.legend()
        plt.show()
else:
    print("Skipped: no model prediction was produced.")

## 6. Run a complete robot episode

ABC now generates a **new simulated scene**. It does not reconstruct the exact real scene in the demonstration. The policy observes, predicts an action chunk, executes 15 actions, observes again, and continues for up to 236 chunks.

For this task, ABC reports whether all bottles were inside the bin **at any observed point**, and separately whether that condition held at the end. Both are shown. This is simulation evidence, not physical-robot validation.

Every seed starts a fresh process/environment. This avoids carrying the previous episode's arm pose into the next reset. The notebook uses sequential CPU MuJoCo physics with MJWarp rendering and synchronous policy execution throughout. Keep the protocol fixed when comparing checkpoints. [ABC environment and evaluation](https://github.com/amazon-far/abc/blob/d0e987b61b376f1a1b8777b19149f190b2ba6939/abc_sim/README.md)

In [ ]:
def evaluate_one(checkpoint, seed, prompt, label):
    output = SESSION / label / f"seed-{seed}-{uuid.uuid4().hex[:8]}"
    output.mkdir(parents=True, exist_ok=False)
    args = evaluation_args(checkpoint, output, seed=seed, prompt=prompt)
    abc_command(args, output / "execution.log")
    result = read_result(output)
    result["notebook_output"] = str(output)
    result["checkpoint_sha256"] = sha256(checkpoint)
    return result


if RUN_GPU:
    BASELINE, CANDIDATE = [], []
    STAGES["fine_tuning"] = "not_started" if RUN_FINE_TUNING else "skipped"
    with tracked_stage(STAGES, "evaluation"):
        BASELINE = [evaluate_one(PARENT, EVAL_SEEDS[0], PARENT_PROMPT, "baseline")]
        world = BASELINE[0]["worlds"][0]
        print("Goal reached during episode:", world["success"])
        print("Goal satisfied at end:", world["final_success"])
        print("Maximum task progress:", world.get("max_reward", "Not reported"))
        if world.get("video_path"):
            display(Video(filename=world["video_path"], embed=True, width=700))
    if len(BASELINE) < len(EVAL_SEEDS):
        STAGES["evaluation"] = "partial"
else:
    print("Skipped: no simulator outcomes were measured.")

## 7. Repeat and inspect the outcomes

Three scenarios make a readable hello-world, not a reliable performance estimate. Increase the sample and predeclare a separate final test set for a real study. An action chunk is a step; **one complete episode is one trial**.

No LLM or fabricated score fills missing outcomes. A failed process stops the cell, and a result is accepted only with a successful command receipt and actual evaluator fields.

In [ ]:
if RUN_GPU:
    CANDIDATE = []
    STAGES["fine_tuning"] = "not_started" if RUN_FINE_TUNING else "skipped"
    with tracked_stage(STAGES, "evaluation"):
        assert BASELINE, "Run the complete-episode cell first."
        BASELINE = BASELINE[:1]
        for seed in EVAL_SEEDS[1:]:
            BASELINE.append(evaluate_one(PARENT, seed, PARENT_PROMPT, "baseline"))
        worlds = [result["worlds"][0] for result in BASELINE]
        successes = sum(world["success"] for world in worlds)
        display(Markdown(f"### {successes} / {len(worlds)} episodes reached the goal"))
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.bar([str(w["world_seed"]) for w in worlds], [int(w["success"]) for w in worlds])
        ax.set(
            xlabel="Actual world seed",
            ylabel="Goal reached",
            yticks=[0, 1],
            title="Measured simulator outcomes",
        )
        plt.show()
        for result in BASELINE:
            world = result["worlds"][0]
            print(
                {
                    "seed": world["world_seed"],
                    "reached_goal": world["success"],
                    "goal_at_end": world["final_success"],
                    "steps": world["steps"],
                    "video": world.get("video_path"),
                }
            )
        write_json(SESSION / "baseline.json", BASELINE)
else:
    print("No outcome chart: GPU evaluation was not run.")

## 8. Optional: fine-tune, save and evaluate a new checkpoint

Enable `RUN_FINE_TUNING=True` in the configuration cell and re-run from there. The training input is the preview's **real plus simulation** training folders; validation remains separate. We use ABC's bottles mixture (81.72% real / 18.28% sim), inherited checkpoint normalization, and its default learning-rate warmup. Twenty steps on a tiny preview only tests the mechanics and barely enters warmup; it may make performance worse.

Full VLA fine-tuning needs substantially more memory than inference. Batch size 1 is not a claim that a small GPU is sufficient. `TRAIN_GPUS>1` enables ABC's FSDP path. Inspect memory use before extending the budget. W&B upload is disabled.

A fresh output directory and `ckpt_every=TRAIN_STEPS` ensure this cell evaluates its own newly saved checkpoint. The local checkpoint's prompt is derived from the actual sim training metadata; if it differs from the parent's, the comparison explicitly reports that second change.

In [ ]:
if RUN_FINE_TUNING:
    CANDIDATE = []
    with tracked_stage(STAGES, "fine_tuning"):
        # Use the trainer's own prompt resolution, including legacy sim_ aliases.
        prompt_script = SESSION / "training-prompts.py"
        prompt_script.write_text(
            textwrap.dedent("""
    import json, sys
    from pathlib import Path
    from abc_minimal.config import VLAModelConfig
    from abc_minimal.dataloader import scan_episodes
    from abc_minimal.dit import task_name_to_prompt
    rows = scan_episodes(Path(sys.argv[1]), "", VLAModelConfig())
    prompts = sorted({row[5] or task_name_to_prompt(row[4]) for row in rows})
    Path(sys.argv[2]).write_text(json.dumps(prompts))
    """)
        )
        abc_command(
            [prompt_script, CACHE / "train_sim", SESSION / "training-prompts.json"],
            SESSION / "training-prompts.log",
        )
        sim_prompts = set(json.loads((SESSION / "training-prompts.json").read_text()))
        assert len(sim_prompts) == 1 and all(sim_prompts), (
            "This hello-world expects one recorded simulator task prompt."
        )
        TRAINED_PROMPT = next(iter(sim_prompts))
        training_output = SESSION / "training" / uuid.uuid4().hex[:8]
        training_output.mkdir(parents=True, exist_ok=False)
        training_args = [
            "train.py",
            "--policy",
            "vla",
            "--load-pretrained",
            "--pretrained-ckpt-name",
            PARENT_NAME,
            "--cache-root",
            str(CACHE),
            "--mixture-preset",
            "bottles",
            "--output-dir",
            str(training_output),
            "--batch-size",
            str(TRAIN_BATCH_SIZE),
            "--num-workers",
            "0",
            "--train-steps",
            str(TRAIN_STEPS),
            "--ckpt-every",
            str(TRAIN_STEPS),
            "--log-every",
            "1",
            "--val-every",
            str(TRAIN_STEPS),
            "--val-batches",
            "1",
            "--no-compile",
            "--no-log-wandb",
        ]
        if TRAIN_GPUS > 1:
            training_args = [
                "-m",
                "torch.distributed.run",
                "--standalone",
                "--nproc-per-node",
                str(TRAIN_GPUS),
                *training_args,
                "--fsdp",
            ]
        abc_command(training_args, training_output / "training.log")
        FINETUNED = training_output / f"{TRAIN_STEPS}.pt"
        assert FINETUNED.is_file(), "Training did not save the expected fresh checkpoint."
        import re

        loss_points = re.findall(
            r"step\s+(\d+)\s+loss\s+([\d.]+)", (training_output / "training.log").read_text()
        )
        if loss_points:
            fig, ax = plt.subplots(figsize=(9, 3))
            ax.plot([int(s) for s, _ in loss_points], [float(v) for _, v in loss_points])
            ax.set(
                xlabel="Training step",
                ylabel="Diffusion loss",
                title="Training mechanics · not task success",
            )
            plt.show()
        CANDIDATE = [
            evaluate_one(FINETUNED, seed, TRAINED_PROMPT, "finetuned") for seed in EVAL_SEEDS
        ]
        notes = comparison_notes(BASELINE, CANDIDATE)
        for note in notes:
            print("Comparison condition:", note)
        if not notes:
            print(
                "Recorded comparison conditions match; this small sample still does not establish a reliable gain."
            )
        fig, ax = plt.subplots(figsize=(7, 3))
        counts = [sum(r["worlds"][0]["success"] for r in group) for group in (BASELINE, CANDIDATE)]
        ax.bar(["Original", f"Fine-tuned ({TRAIN_STEPS} steps)"], counts)
        ax.set(
            ylabel="Episodes reaching goal",
            ylim=(0, len(EVAL_SEEDS)),
            title="Descriptive before / after",
        )
        plt.show()
        write_json(SESSION / "candidate.json", CANDIDATE)
else:
    print("Skipped: no training took place and no fine-tuned result is claimed.")

## 9. Keep the evidence

The session folder contains the exact commands, their exit status, prepared-data hashes, observation clip, and any actual model predictions, simulator summaries and rollout videos. Failed or skipped GPU steps do not become zero-success trials.

Training uses the training split; offline diagnostics use validation data. Do not use these hello-world evaluation seeds repeatedly to select a winner and then present them as an untouched final test. The parent checkpoint may already have seen these tasks.

Save the exported evidence locally. **Clear notebook outputs before committing or sharing**: outputs can contain your local paths, videos and later customer data. The repository version intentionally contains no execution outputs, tokens or model weights.

In [ ]:
provenance = {
    "format": "abc-xdof-notebook/v1",
    "abc_revision": revision,
    "task": TASK,
    "data_source": "ABC prepared bottles preview (real + simulation)",
    "data_manifest": "data-manifest.json",
    "episode": EPISODE.name,
    "frame": FRAME,
    "python": platform.python_version(),
    "notebook_packages": {
        name: importlib.metadata.version(name) for name in ["numpy", "matplotlib", "nbformat"]
    },
    "inference": STAGES["inference"],
    "evaluation": STAGES["evaluation"],
    "fine_tuning": STAGES["fine_tuning"],
    "requested_seeds": EVAL_SEEDS,
    "baseline_episodes": len(BASELINE),
    "candidate_episodes": len(CANDIDATE),
}
if RUN_GPU:
    provenance["parent_sha256"] = sha256(PARENT)
    provenance["parent_metadata_sha256"] = sha256(PARENT.with_suffix(".json"))
    # uv records ABC's resolved packages without installing pip into its environment.
    run_command(
        ["uv", "pip", "freeze", "--python", str(ABC_PYTHON)],
        cwd=ABC,
        log=SESSION / "abc-packages.log",
    )
write_json(SESSION / "provenance.json", provenance)
print(json.dumps(provenance, indent=2))
print("Evidence folder:", SESSION)

## What to try next

1. Replace the preview with a deliberately selected larger task dataset. ABC's `scripts/export_hf_task.py` downloads/converts raw XDOF MCAPs after you accept the Hugging Face access conditions. Use a new cache and preserve the raw repository revision and episode split.
2. Run longer fine-tuning with a measured memory/compute budget and evaluate held-out scenarios. Track task completion and failure videos alongside learning curves.
3. Add a second compatible model only after validating its observation/action mapping. A different customer's robot needs its own compatible data and evaluation environment.

**Sources:** [ABC release](https://github.com/amazon-far/abc/tree/d0e987b61b376f1a1b8777b19149f190b2ba6939) · [XDOF dataset card](https://huggingface.co/datasets/XDOF/ABC-130k) · [Training configuration](https://github.com/amazon-far/abc/blob/d0e987b61b376f1a1b8777b19149f190b2ba6939/abc_minimal/config.py) · [Simulator evaluation](https://github.com/amazon-far/abc/blob/d0e987b61b376f1a1b8777b19149f190b2ba6939/abc_minimal/eval_policy.py)